In [12]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
import lightning as L
import yaml
import lightning_scripts.lightning_classifier_matched_speech_in_noise as lightning 
import importlib
import torch
from lightning_scripts import jsinV3DataLoader_precombined_batched 
import pandas as pd 

In [13]:
torch.set_float32_matmul_precision = 'medium'

In [14]:
###
config_path = "model_configs/word_speaker_audioset_resnet50_MatchedSpeechInNoiseDatasetBatched.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)
config['num_workers'] = 0
config['hparas']['batch_size'] = 192

### Make sure dataset labels match examples

In [15]:
importlib.reload(jsinV3DataLoader_precombined_batched)

MatchedSpeechInNoiseDatasetBatched = jsinV3DataLoader_precombined_batched.MatchedSpeechInNoiseDatasetBatched
dataset = MatchedSpeechInNoiseDatasetBatched(config['data']['speech_h5_path'],
                                             config['data']['val_noise_h5_path'],
                                             target_keys=config['data']['target_keys'],
                                             batch_size=4)

output_11, output_12, output_21, output_22, target_11, target_12, target_21, target_22 = dataset[0]

In [16]:
class_map, full_map = dataset.class_map()

In [17]:
SR = 20_000
pair_ix = 3
word_1 = class_map[int(target_11['signal/word_int'][pair_ix])]
word_2 = class_map[int(target_21['signal/word_int'][pair_ix])]


print(word_1)
display(Audio(output_11[pair_ix], rate=SR))
display(Audio(output_12[pair_ix], rate=SR))
print(word_2)
display(Audio(output_21[pair_ix], rate=SR))
display(Audio(output_22[pair_ix], rate=SR))

operation


activity


### Try lightning module

In [18]:
importlib.reload(lightning)
LitWordAudioSetModel = lightning.LitWordAudioSetModel

model = LitWordAudioSetModel(config)

In [19]:
model

LitWordAudioSetModel(
  (audio_rep): AudioToAudioRepresentation(
    (rep): AudioToCochleagram(
      (envelope_extraction): HilbertEnvelopeExtraction()
      (downsampling_op): SincWithKaiserWindow()
      (Cochleagram): Cochleagram(
        (compute_subbands): ComputeSubbands()
        (envelope_extraction): HilbertEnvelopeExtraction()
        (downsampling): SincWithKaiserWindow()
      )
    )
    (compression): ClippedGradPower(
      (compression_function): ClippedGradPowerCompression()
    )
  )
  (model): ModelWithFrontEnd(
    (front_end): AudioToAudioRepresentation(
      (rep): AudioToCochleagram(
        (envelope_extraction): HilbertEnvelopeExtraction()
        (downsampling_op): SincWithKaiserWindow()
        (Cochleagram): Cochleagram(
          (compute_subbands): ComputeSubbands()
          (envelope_extraction): HilbertEnvelopeExtraction()
          (downsampling): SincWithKaiserWindow()
        )
      )
      (compression): ClippedGradPower(
        (compression_fun

In [20]:
# model

In [21]:
trainer = L.Trainer(
    limit_train_batches=5,
    limit_val_batches=2,
    max_epochs=config['hparas']['epochs'],
    strategy='ddp_notebook',
    devices=1)


/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3. ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [22]:
trainer.fit(model)

You are using a CUDA device ('NVIDIA H100 PCIe') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name            | Type                       | Params | Mode 
-----------------------------------------------------------------------
0 | audio_rep       | AudioToAudioRepresentation | 0      | train
1 | model           | ModelWithFrontEnd          |

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:298: The number of training batches (5) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined